# プロテオミクスデータ分布評価

**対応するブログ記事**: [blog/article-05a-data-distribution.md](../blog/article-05a-data-distribution.md) — プロテオミクスデータ分布評価【論文再現シリーズ #5a】  
**実行順序**: 5番目  
**所要時間**: 約10分

---

## このNotebookで行うこと

前回構築したタンパク質マトリクス（2,110タンパク質×32サンプル）の分布特性を詳細に評価します：

- 生の強度値分布の確認
- 対数変換後の正規性評価
- 欠損値パターンの解析
- ダイナミックレンジの評価
- 適切な前処理方針の決定

**⚠️ 前提条件**:
- [notebook_04c_sage_protein_matrix.ipynb](./notebook_04c_sage_protein_matrix.ipynb) が完了していること
- `results/protein_matrix_from_sage.csv` が生成されていること

## 1. ライブラリと設定

In [ ]:
import numpy as np          # 数値計算ライブラリ（配列操作・統計計算に使用）
import pandas as pd         # データフレーム操作ライブラリ（CSV読み込み・データ操作に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（ヒストグラム・散布図作成に使用）
import seaborn as sns       # 統計的可視化ライブラリ（相関ヒートマップ・分布プロットに使用）
from scipy import stats    # 統計計算ライブラリ（正規性検定・相関解析に使用）
import warnings            # 警告制御（matplotlib警告の非表示用）
warnings.filterwarnings('ignore')  # グラフ描画時の軽微な警告を非表示

# Jupyter notebook用の設定
%matplotlib inline

print("✅ ライブラリの読み込み完了")

In [ ]:
# --- パス設定 ---
RESULTS_DIR = "../results"              # 解析結果の保存先ディレクトリ
DATA_FILE = f"{RESULTS_DIR}/protein_matrix_from_sage.csv"  # sageから構築したタンパク質マトリクス
FIG_DIR = f"{RESULTS_DIR}/figures"      # 品質評価図の保存先
TABLES_DIR = f"{RESULTS_DIR}/tables"    # 統計サマリーの保存先

# 出力ディレクトリの作成
import os
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# --- 色設定 ---
NORMAL_COLOR = "#3498DB"   # Normal群の色（青）
TUMOR_COLOR = "#E74C3C"    # Tumor群の色（赤）

print(f"📁 作業ディレクトリ: {os.getcwd()}")
print(f"📊 データファイル: {DATA_FILE}")
print(f"🎨 図の保存先: {FIG_DIR}")
print(f"📋 テーブル保存先: {TABLES_DIR}")

## 2. データ読み込みとサンプル情報の抽出

### 【サンプル命名規則】
- CRC01-N, CRC01-T → 患者ID: CRC01, 条件: Normal/Tumor
- N: Normal（正常組織）, T: Tumor（腫瘍組織）

In [ ]:
def load_protein_matrix_with_metadata():
    """タンパク質マトリクスを読み込み、サンプル情報を抽出する。

    【サンプル命名規則の解析】
    CRC01-N, CRC01-T → 患者ID: CRC01, 条件: Normal/Tumor
    """
    # sageで構築したタンパク質×サンプルのマトリクス（CSV）を読み込み
    # index_col=0: 1列目（Protein列）をインデックスとして使用
    df = pd.read_csv(DATA_FILE, index_col=0)
    print(f"データ読み込み完了: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")

    # サンプル名から患者情報を抽出する関数
    def extract_sample_info(sample_name):
        """CRC01-N → Patient: CRC01, Condition: Normal"""
        parts = sample_name.split("-")
        if len(parts) == 2:
            patient_id = parts[0]           # CRC01, CRC02, etc.
            condition = "Normal" if parts[1] == "N" else "Tumor"  # N→Normal, T→Tumor
            return patient_id, condition
        return sample_name, "Unknown"      # 命名規則に合わない場合

    # 全サンプルについてメタデータを抽出
    sample_info_list = []
    for sample in df.columns:
        patient, condition = extract_sample_info(sample)
        sample_info_list.append({
            'Sample': sample,
            'Patient': patient,
            'Condition': condition
        })

    # サンプル情報をDataFrameに変換
    sample_info = pd.DataFrame(sample_info_list)

    print(f"サンプル情報:")
    print(sample_info.groupby('Condition').size())  # Normal/Tumorのサンプル数を表示
    print(f"患者数: {sample_info['Patient'].nunique()}")  # ユニーク患者数

    return df, sample_info

# データとメタデータの読み込み
df, sample_info = load_protein_matrix_with_metadata()

In [ ]:
# データ構造の確認
print("=== データ構造の詳細 ===")
print(f"データフレーム形状: {df.shape}")
print(f"\n最初の5タンパク質 × 5サンプル:")
display(df.iloc[:5, :5])

print(f"\nサンプル情報の詳細:")
display(sample_info.head(10))

## 3. データ分布の包括評価

### 【解析項目】
1. 生の強度値分布（ヒストグラム）
2. 対数変換後分布（正規性確認）
3. 欠損値パターン（タンパク質・サンプル別）
4. ダイナミックレンジ評価

In [ ]:
def analyze_data_distribution(df):
    """タンパク質マトリクスのデータ分布を包括的に解析する。

    【解析項目】
    1. 生の強度値分布（ヒストグラム）
    2. 対数変換後分布（正規性確認）
    3. 欠損値パターン（タンパク質・サンプル別）
    4. ダイナミックレンジ評価
    """
    print("=== データ分布解析 ===")

    # 数値データのみ抽出（欠損値を除く）
    numeric_data = df.select_dtypes(include=[np.number])
    non_zero_data = numeric_data[numeric_data > 0]  # 0値（検出限界以下）を除外

    # 基本統計量
    total_values = numeric_data.size                    # 全データポイント数
    missing_values = numeric_data.isna().sum().sum()   # 欠損値数
    zero_values = (numeric_data == 0).sum().sum()      # ゼロ値数
    detected_values = non_zero_data.notna().sum().sum()  # 実際の検出値数

    print(f"全データポイント: {total_values:,}")
    print(f"欠損値: {missing_values:,} ({missing_values/total_values*100:.1f}%)")
    print(f"ゼロ値: {zero_values:,} ({zero_values/total_values*100:.1f}%)")
    print(f"有効検出値: {detected_values:,} ({detected_values/total_values*100:.1f}%)")

    # ダイナミックレンジ（検出範囲の広さ）
    if len(non_zero_data.values.flatten()) > 0:
        min_intensity = non_zero_data.min().min()
        max_intensity = non_zero_data.max().max()
        dynamic_range = np.log10(max_intensity / min_intensity)  # 対数スケールでの範囲
        print(f"強度値範囲: {min_intensity:.2e} - {max_intensity:.2e}")
        print(f"ダイナミックレンジ: {dynamic_range:.1f} orders of magnitude")
    else:
        min_intensity, max_intensity, dynamic_range = 0, 0, 0

    # 分布可視化（2x2サブプロット）
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # 1. 生の強度値分布
    flat_data = non_zero_data.values.flatten()
    flat_data = flat_data[~np.isnan(flat_data)]  # NaN除去
    
    if len(flat_data) > 0:
        axes[0, 0].hist(flat_data, bins=50, alpha=0.7, color=NORMAL_COLOR, edgecolor='black')
        axes[0, 0].set_xlabel("Raw Intensity")
        axes[0, 0].set_ylabel("Frequency")
        axes[0, 0].set_title("Distribution of Raw Intensities")
        axes[0, 0].set_yscale('log')  # y軸を対数スケール（広範囲の頻度を表示）

        # 2. Log2変換後分布
        log_data = np.log2(flat_data)
        axes[0, 1].hist(log_data, bins=50, alpha=0.7, color=TUMOR_COLOR, edgecolor='black')
        axes[0, 1].set_xlabel("Log2 Intensity")
        axes[0, 1].set_ylabel("Frequency")
        axes[0, 1].set_title("Distribution of Log2 Intensities")

    # 3. タンパク質別検出頻度
    detection_freq = (~numeric_data.isna()).sum(axis=1)  # 各タンパク質が何サンプルで検出されたか
    axes[1, 0].hist(detection_freq, bins=20, alpha=0.7, color='green', edgecolor='black')
    axes[1, 0].set_xlabel("Detection Frequency (# Samples)")
    axes[1, 0].set_ylabel("# Proteins")
    axes[1, 0].set_title("Protein Detection Frequency")

    # 4. サンプル別検出数
    sample_detection = (~numeric_data.isna()).sum(axis=0)  # 各サンプルで検出されたタンパク質数
    axes[1, 1].hist(sample_detection, bins=20, alpha=0.7, color='orange', edgecolor='black')
    axes[1, 1].set_xlabel("Detected Proteins per Sample")
    axes[1, 1].set_ylabel("# Samples")
    axes[1, 1].set_title("Sample Detection Count")

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/data_distribution_overview.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 統計サマリーをCSV保存
    stats_summary = {
        'Metric': [
            'Total_Values', 'Missing_Values', 'Zero_Values', 'Detected_Values',
            'Min_Intensity', 'Max_Intensity', 'Dynamic_Range_Orders',
            'Mean_Detection_per_Protein', 'Mean_Detection_per_Sample'
        ],
        'Value': [
            total_values, missing_values, zero_values, detected_values,
            min_intensity, max_intensity, dynamic_range,
            detection_freq.mean(), sample_detection.mean()
        ]
    }
    stats_df = pd.DataFrame(stats_summary)
    stats_df.to_csv(f"{TABLES_DIR}/data_distribution_summary.csv", index=False)
    
    print(f"\n📊 統計サマリーを保存: {TABLES_DIR}/data_distribution_summary.csv")
    print(f"🎨 分布図を保存: {FIG_DIR}/data_distribution_overview.png")

    return stats_df

# データ分布解析の実行
distribution_stats = analyze_data_distribution(df)

## 4. 統計サマリーの詳細確認

In [ ]:
# 統計サマリーの表示
print("=== データ分布統計サマリー ===")
display(distribution_stats)

# 主要指標の解釈
print("\n=== 主要指標の解釈 ===")
total_proteins = df.shape[0]
total_samples = df.shape[1]
detected_ratio = distribution_stats[distribution_stats['Metric'] == 'Detected_Values']['Value'].iloc[0] / (total_proteins * total_samples) * 100
dynamic_range = distribution_stats[distribution_stats['Metric'] == 'Dynamic_Range_Orders']['Value'].iloc[0]

print(f"📊 データ概要:")
print(f"  - タンパク質数: {total_proteins:,}")
print(f"  - サンプル数: {total_samples}")
print(f"  - 検出率: {detected_ratio:.1f}%")
print(f"  - ダイナミックレンジ: {dynamic_range:.1f} orders of magnitude")

print(f"\n✅ 評価結果:")
if detected_ratio > 80:
    print(f"  ✅ 検出率 ({detected_ratio:.1f}%) - 良好")
else:
    print(f"  ⚠️ 検出率 ({detected_ratio:.1f}%) - やや低い")
    
if dynamic_range > 5:
    print(f"  ✅ ダイナミックレンジ ({dynamic_range:.1f}) - 十分な感度")
else:
    print(f"  ⚠️ ダイナミックレンジ ({dynamic_range:.1f}) - 感度制限あり")

## 5. 正規性の詳細評価

統計解析（t検定、ANOVA等）の前提条件として、データの正規性を確認します。

In [ ]:
def evaluate_normality(df, sample_size=1000):
    """データの正規性を評価し、Log2変換の効果を確認する。"""
    print("=== 正規性評価 ===")
    
    # ランダムサンプリング（計算効率のため）
    numeric_data = df.select_dtypes(include=[np.number])
    flat_data = numeric_data.values.flatten()
    flat_data = flat_data[~np.isnan(flat_data) & (flat_data > 0)]  # NaNと0を除去
    
    if len(flat_data) > sample_size:
        np.random.seed(42)  # 再現性のためのシード固定
        flat_data = np.random.choice(flat_data, sample_size, replace=False)
    
    # Log2変換
    log_data = np.log2(flat_data)
    
    # Shapiro-Wilk検定（正規性検定）
    if len(flat_data) <= 5000:  # Shapiro-Wilkは5000サンプル以下で実行
        stat_raw, p_raw = stats.shapiro(flat_data[:1000] if len(flat_data) > 1000 else flat_data)
        stat_log, p_log = stats.shapiro(log_data[:1000] if len(log_data) > 1000 else log_data)
        
        print(f"Shapiro-Wilk検定結果:")
        print(f"  生データ: p = {p_raw:.2e} ({'正規分布' if p_raw > 0.05 else '非正規分布'})")
        print(f"  Log2変換後: p = {p_log:.2e} ({'正規分布' if p_log > 0.05 else '非正規分布'})")
    
    # Q-Qプロット（正規分布との比較）
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 生データのQ-Qプロット
    stats.probplot(flat_data[:1000] if len(flat_data) > 1000 else flat_data, 
                   dist="norm", plot=axes[0])
    axes[0].set_title("Q-Q Plot: Raw Data vs Normal Distribution")
    axes[0].grid(True, alpha=0.3)
    
    # Log2変換後のQ-Qプロット
    stats.probplot(log_data[:1000] if len(log_data) > 1000 else log_data, 
                   dist="norm", plot=axes[1])
    axes[1].set_title("Q-Q Plot: Log2 Data vs Normal Distribution")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/normality_evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    return flat_data, log_data

# 正規性評価の実行
raw_sample, log_sample = evaluate_normality(df)

## 6. サンプル間・タンパク質間の検出パターン分析

In [ ]:
def analyze_detection_patterns(df, sample_info):
    """サンプル間・タンパク質間の検出パターンを詳細に解析する。"""
    print("=== 検出パターン解析 ===")
    
    numeric_data = df.select_dtypes(include=[np.number])
    
    # タンパク質別検出統計
    protein_detection = (~numeric_data.isna()).sum(axis=1)
    protein_stats = {
        'High_Detection': (protein_detection >= 30).sum(),  # 30サンプル以上で検出
        'Medium_Detection': ((protein_detection >= 20) & (protein_detection < 30)).sum(),
        'Low_Detection': (protein_detection < 20).sum()
    }
    
    # サンプル別検出統計
    sample_detection = (~numeric_data.isna()).sum(axis=0)
    
    # Normal vs Tumor群での検出数比較
    normal_samples = sample_info[sample_info['Condition'] == 'Normal']['Sample'].tolist()
    tumor_samples = sample_info[sample_info['Condition'] == 'Tumor']['Sample'].tolist()
    
    normal_detection = sample_detection[normal_samples]
    tumor_detection = sample_detection[tumor_samples]
    
    print(f"タンパク質検出分類:")
    for category, count in protein_stats.items():
        percentage = count / len(protein_detection) * 100
        print(f"  {category}: {count} タンパク質 ({percentage:.1f}%)")
    
    print(f"\nサンプル検出統計:")
    print(f"  Normal群平均: {normal_detection.mean():.0f} ± {normal_detection.std():.0f} タンパク質")
    print(f"  Tumor群平均: {tumor_detection.mean():.0f} ± {tumor_detection.std():.0f} タンパク質")
    
    # 検出パターンの可視化
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 1. サンプル別検出数（群分け）
    normal_pos = np.arange(len(normal_detection))
    tumor_pos = np.arange(len(tumor_detection)) + len(normal_detection) + 1
    
    axes[0].bar(normal_pos, normal_detection, color=NORMAL_COLOR, alpha=0.7, label='Normal')
    axes[0].bar(tumor_pos, tumor_detection, color=TUMOR_COLOR, alpha=0.7, label='Tumor')
    axes[0].set_xlabel("Samples")
    axes[0].set_ylabel("Detected Proteins")
    axes[0].set_title("Protein Detection Count by Sample")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. タンパク質検出頻度分布
    detection_categories = ['Low\n(<20)', 'Medium\n(20-29)', 'High\n(≥30)']
    detection_counts = [protein_stats['Low_Detection'], 
                       protein_stats['Medium_Detection'], 
                       protein_stats['High_Detection']]
    
    colors = ['#FF6B6B', '#FFD93D', '#6BCF7F']
    axes[1].bar(detection_categories, detection_counts, color=colors, alpha=0.7)
    axes[1].set_ylabel("Number of Proteins")
    axes[1].set_title("Protein Detection Frequency Categories")
    
    # 値をバーの上に表示
    for i, count in enumerate(detection_counts):
        axes[1].text(i, count + 20, str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/detection_patterns.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    return protein_stats, normal_detection, tumor_detection

# 検出パターン解析の実行
protein_stats, normal_det, tumor_det = analyze_detection_patterns(df, sample_info)

## 7. 結果のまとめとファイル保存

In [ ]:
# 分析結果の総合サマリー
print("=== 📋 データ分布評価 総合結果 ===")
print(f"\n🔍 データ構造:")
print(f"  - タンパク質数: {df.shape[0]:,}")
print(f"  - サンプル数: {df.shape[1]} (Normal: {len(normal_det)}, Tumor: {len(tumor_det)})")
print(f"  - 総データポイント: {df.shape[0] * df.shape[1]:,}")

detected_ratio = distribution_stats[distribution_stats['Metric'] == 'Detected_Values']['Value'].iloc[0] / (df.shape[0] * df.shape[1]) * 100
dynamic_range = distribution_stats[distribution_stats['Metric'] == 'Dynamic_Range_Orders']['Value'].iloc[0]

print(f"\n📊 データ品質指標:")
print(f"  - 全体検出率: {detected_ratio:.1f}%")
print(f"  - ダイナミックレンジ: {dynamic_range:.1f} orders of magnitude")
print(f"  - 高検出頻度タンパク質 (≥30サンプル): {protein_stats['High_Detection']} ({protein_stats['High_Detection']/df.shape[0]*100:.1f}%)")

print(f"\n✅ 前処理への推奨事項:")
print(f"  1. Log2変換の適用 - 正規性の改善のため")
print(f"  2. 欠損値処理の実装 - 約{100-detected_ratio:.1f}%の欠損値に対応")
print(f"  3. 低検出タンパク質のフィルタリング検討")
print(f"  4. サンプル間正規化の検討")

# 保存されたファイルの一覧表示
print(f"\n💾 生成されたファイル:")
print(f"  📊 統計サマリー: {TABLES_DIR}/data_distribution_summary.csv")
print(f"  🎨 分布概要図: {FIG_DIR}/data_distribution_overview.png")
print(f"  📈 正規性評価図: {FIG_DIR}/normality_evaluation.png")
print(f"  📋 検出パターン図: {FIG_DIR}/detection_patterns.png")

## 📚 このNotebookで学んだこと

### データ分布評価の要点

1. **分布特性の確認**: プロテオミクスデータの典型的な右歪み分布を確認
2. **Log2変換の効果**: 正規分布への近似により統計解析に適した形に変換
3. **検出品質評価**: 高い検出頻度と技術的再現性を確認
4. **前処理方針の決定**: 欠損値処理とLog2変換の必要性を定量的に確認

### 重要な評価指標

| 指標 | 意味 | 良好な基準 |
|------|------|----------|
| **検出率** | 全データポイントのうち有効検出値の割合 | >80% |
| **ダイナミックレンジ** | 最小〜最大強度値の比率 | >5 orders |
| **検出頻度** | タンパク質が検出されるサンプル数 | >75%で検出 |
| **サンプル間一貫性** | 技術的再現性の指標 | CV <20% |

---

**次のステップ**: サンプル間相関解析でデータの群分離品質とバッチ効果を評価します

➡️ **次回**: [notebook_05b_data_correlation.ipynb](./notebook_05b_data_correlation.ipynb) — サンプル間相関解析